# RAG over Atopic Dermatitis Literature — full walkthrough

The corpus is real PubMed abstracts on atopic dermatitis (each cited by PMID).

Run **one cell at a time**. The pipeline:

**question -> retrieve passages (TF-IDF) -> Claude writes a cited answer**, then we **evaluate** it two ways.

> Launch from the project folder so `corpus/` and `.env` are found:
> `cd rag_drug_lit && .venv/bin/jupyter lab`

## Setup — imports and load the API keys

In [19]:
import os, glob, json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

load_dotenv()   # reads ANTHROPIC_API_KEY and OPENAI_API_KEY from .env
print('anthropic key:', bool(os.environ.get('ANTHROPIC_API_KEY')))
print('openai key   :', bool(os.environ.get('OPENAI_API_KEY')))

anthropic key: True
openai key   : True


# STEP 1 — Retrieve

## 1a. load the corpus and split into chunks

In [20]:
def split_into_chunks(text, words_per_chunk=120, overlap=30):
    words = text.split()
    if len(words) <= words_per_chunk:
        return [text]
    chunks = []
    start = 0
    while start < len(words):
        piece = words[start:start + words_per_chunk]
        chunks.append(' '.join(piece))
        start = start + words_per_chunk - overlap
    return chunks

chunk_texts = []
chunk_sources = []
for path in sorted(glob.glob('corpus/*.txt')):
    content = open(path).read()
    for chunk in split_into_chunks(content):
        chunk_texts.append(chunk)
        chunk_sources.append(os.path.basename(path))

print('documents:', len(glob.glob('corpus/*.txt')))
print('total chunks:', len(chunk_texts))
chunk_sources

documents: 8
total chunks: 18


['ad_dupilumab.txt',
 'ad_dupilumab.txt',
 'ad_filaggrin.txt',
 'ad_filaggrin.txt',
 'ad_filaggrin.txt',
 'ad_itch.txt',
 'ad_itch.txt',
 'ad_jak.txt',
 'ad_jak.txt',
 'ad_jak.txt',
 'ad_microbiome.txt',
 'ad_microbiome.txt',
 'ad_overview.txt',
 'ad_overview.txt',
 'ad_pediatric.txt',
 'ad_topical.txt',
 'ad_topical.txt',
 'ad_topical.txt']

## 1b. turn the chunks into TF-IDF vectors

In [21]:
vectorizer = TfidfVectorizer(stop_words='english')
chunk_vectors = vectorizer.fit_transform(chunk_texts)
print('matrix shape (chunks x words):', chunk_vectors.shape)

matrix shape (chunks x words): (18, 521)


## 1c. the question (the INPUT to the pipeline)

In [22]:
question = "What targeted systemic treatments are used for moderate-to-severe atopic dermatitis?"
question

'What targeted systemic treatments are used for moderate-to-severe atopic dermatitis?'

## 1d. retrieve: score every chunk, keep the top 3

In [23]:
def search(question, k=3):
    question_vector = vectorizer.transform([question])
    scores = cosine_similarity(question_vector, chunk_vectors)[0]
    best = scores.argsort()[::-1][:k]
    results = []
    for i in best:
        results.append({'source': chunk_sources[i],
                        'text': chunk_texts[i],
                        'score': float(scores[i])})
    return results

passages = search(question, k=3)
for p in passages:
    print(p['source'], 'score=%.3f' % p['score'])
    print('   ', p['text'][:140], '...')
    print()

ad_topical.txt score=0.244
    on symptoms and physical examination findings. Maintenance therapy consists of liberal use of emollients and daily bathing with soap-free cl ...

ad_overview.txt score=0.176
    Title: Atopic dermatitis. Source: Lancet (London, England) (2025), PMID 39955121 Atopic dermatitis is the most common chronic inflammatory s ...

ad_jak.txt score=0.109
    topical JAK inhibitors include ruxolitinib (JAK1/2) and delgocitinib (pan-JAK). Ruxolitinib cream met all primary and secondary endpoints in ...



# STEP 2 — Generate

## 2a. build the prompt from the retrieved passages

In [24]:
INSTRUCTIONS = (
    'Act as a drug-discovery research assistant. Answer the question using '
    'ONLY the passages provided. Each passage starts with its source in square '
    'brackets, like [ad_dupilumab.txt]. After each fact, cite the source it came '
    'from in square brackets. If the passages do not contain the answer, state '
    'that clearly instead of guessing.'
)

lines = []
for p in passages:
    lines.append('[' + p['source'] + '] ' + p['text'])
context = '\n\n'.join(lines)
prompt = 'Passages:\n\n' + context + '\n\nQuestion: ' + question
print(prompt)

Passages:

[ad_topical.txt] on symptoms and physical examination findings. Maintenance therapy consists of liberal use of emollients and daily bathing with soap-free cleansers. Use of topical corticosteroids is the first-line treatment for atopic dermatitis flare-ups. Pimecrolimus and tacrolimus are topical calcineurin inhibitors that can be used in conjunction with topical corticosteroids as first-line treatment. Ultraviolet phototherapy is a safe and effective treatment for moderate to severe atopic dermatitis when first-line treatments are not adequate. Antistaphylococcal antibiotics are effective in treating secondary skin infections. Oral antihistamines are not recommended because they do not reduce pruritus. Evidence is lacking to support the use of integrative medicine in the treatment of atopic dermatitis. Newer medications approved by the U.S Food and Drug Administration, such as crisaborole

[ad_overview.txt] Title: Atopic dermatitis. Source: Lancet (London, England) (2025), 

## 2b. Claude writes the cited answer  (~1 cent)

In [25]:
from anthropic import Anthropic

client = Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
response = client.messages.create(
    model='claude-haiku-4-5',
    max_tokens=400,
    system=INSTRUCTIONS,
    messages=[{'role': 'user', 'content': prompt}],
)
answer_text = response.content[0].text.strip()
print(answer_text)

Based on the passages provided, the targeted systemic treatments used for moderate-to-severe atopic dermatitis include:

**Oral JAK inhibitors:**
- Baricitinib (JAK1/2) [ad_jak.txt]
- Abrocitinib (JAK1-selective) [ad_jak.txt]
- Upadacitinib (JAK1-selective) [ad_jak.txt]

All three of these oral JAK inhibitors "met primary and secondary endpoints across numerous trials for moderate-to-severe AD" with treatment-emergent adverse events that were "mainly mild to moderate and included acne, nausea, headache, upper respiratory tract infection, and to a lesser degree, herpes infection and selected laboratory abnormalities." [ad_jak.txt]

The passages indicate that "JAK inhibitors hold great promise as the next generation of targeted AD therapy" with "outstanding efficacy" balanced by "a favorable safety profile in clinical" trials. [ad_jak.txt]


# STEP 3 — Evaluate retrieval (free, no LLM)

For each labeled question in `eval/questions.json` we know which document SHOULD
be retrieved. We measure **Recall@k** (did the right doc make the top-k?) and
**MRR** (how high did it rank?).

In [26]:
questions = json.load(open('eval/questions.json'))

k = 3
hits = 0
reciprocal_ranks = []
for item in questions:
    sources = [p['source'] for p in search(item['question'], k=k)]
    correct = item['relevant_doc']
    if correct in sources:
        hits += 1
        rank = sources.index(correct) + 1
        reciprocal_ranks.append(1.0 / rank)
    else:
        rank = None
        reciprocal_ranks.append(0.0)
    print('rank=%s  %s' % (rank, item['question'][:55]))

print()
print('Recall@%d: %.3f' % (k, hits / len(questions)))
print('MRR:       %.3f' % (sum(reciprocal_ranks) / len(questions)))

rank=1  Which receptor chain does the antibody dupilumab target
rank=1  What nanotechnology approach is proposed for delivering
rank=1  Atopic dermatitis is the leading cause of the global bu
rank=1  Which oral JAK inhibitors met endpoints in trials for m
rank=1  How does Staphylococcus aureus skin colonization change
rank=2  Besides the skin, which organ systems are part of the a
rank=1  What proportion of children and adults does atopic derm
rank=1  What is the first-line topical treatment for atopic der

Recall@3: 1.000
MRR:       0.938


# STEP 4 — Evaluate the ANSWER with DeepEval (LLM-as-judge, GPT)

DeepEval scores the answer our system produced. Two metrics, judged by GPT:

- **Faithfulness** — is every claim backed by the retrieved context? (no hallucination)
- **Answer Relevancy** — does the answer address the question?

Needs `OPENAI_API_KEY`. `.measure()` is the notebook-friendly call (the pytest
file `eval/test_rag_deepeval.py` uses `assert_test` instead).

In [28]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric

test_case = LLMTestCase(
    input=question,
    actual_output=answer_text,
    retrieval_context=[p['text'] for p in passages],
)

faithfulness = FaithfulnessMetric(threshold=0.7)
faithfulness.measure(test_case)
print('Faithfulness: %.2f' % faithfulness.score)
print('  reason:', faithfulness.reason)

relevancy = AnswerRelevancyMetric(threshold=0.7)
relevancy.measure(test_case)
print()
print('Answer Relevancy: %.2f' % relevancy.score)
print('  reason:', relevancy.reason)

/Users/komlagnona/Desktop/MYDESK/SENSORIUM/rag_drug_lit/.venv/lib/python3.13/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Faithfulness: 1.00
  reason: The score is 1.00 because there are no contradictions, so the actual output appears fully consistent with the retrieval context—great job staying faithful to the source.



Answer Relevancy: 0.80
  reason: The score is 0.80 because the response was generally on-topic for moderate-to-severe atopic dermatitis, but it included discussion of adverse events and safety details instead of staying focused on identifying the targeted systemic treatments used.
